In [1]:
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.api import VAR
import pandas as pd
import numpy as np

**Setting up the dataset into a pandas dataframe**

In [12]:
df = pd.read_csv("../../data/FRB_H15.csv").dropna()

#rename columns
df.rename(columns={"Series Description": "Date", "Market yield on U.S. Treasury securities at 1-month  constant maturity, quoted on investment basis": "0Y1M", "Market yield on U.S. Treasury securities at 3-month  constant maturity, quoted on investment basis": "0Y3M", "Market yield on U.S. Treasury securities at 6-month  constant maturity, quoted on investment basis": "0Y6M", "Market yield on U.S. Treasury securities at 1-year  constant maturity, quoted on investment basis": "1Y", "Market yield on U.S. Treasury securities at 2-year  constant maturity, quoted on investment basis": "2Y", "Market yield on U.S. Treasury securities at 3-year  constant maturity, quoted on investment basis": "3Y", "Market yield on U.S. Treasury securities at 5-year  constant maturity, quoted on investment basis": "5Y", "Market yield on U.S. Treasury securities at 7-year  constant maturity, quoted on investment basis": "7Y", "Market yield on U.S. Treasury securities at 10-year  constant maturity, quoted on investment basis": "10Y", "Market yield on U.S. Treasury securities at 20-year constant maturity, quoted on investment basis": "20Y", "Market yield on U.S. Treasury securities at 30-year  constant maturity, quoted on investment basis": "30Y"}, inplace=True)
df.rename(columns={"Market yield on U.S. Treasury securities at 20-year  constant maturity, quoted on investment basis": "20Y"}, inplace=True)

#make index to datetime for timeseries
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)

df.tail(30)

,0Y1M,0Y3M,0Y6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
Date,,,,,,,,,,,
2026-04-03,3.71,3.71,3.73,3.72,3.84,3.88,3.99,4.17,4.35,4.91,4.91
2026-04-06,3.72,3.72,3.74,3.72,3.84,3.88,3.98,4.16,4.34,4.89,4.89
2026-04-07,3.68,3.71,3.73,3.68,3.81,3.82,3.95,4.13,4.33,4.90,4.90
2026-04-08,3.67,3.69,3.73,3.69,3.79,3.78,3.92,4.10,4.29,4.87,4.89
2026-04-09,3.66,3.68,3.71,3.68,3.78,3.77,3.91,4.10,4.29,4.88,4.90
2026-04-10,3.67,3.69,3.72,3.70,3.81,3.80,3.94,4.12,4.31,4.89,4.91
2026-04-13,3.69,3.71,3.74,3.70,3.78,3.79,3.92,4.10,4.30,4.88,4.90
2026-04-14,3.71,3.71,3.73,3.71,3.76,3.76,3.87,4.06,4.26,4.84,4.87
2026-04-15,3.72,3.71,3.72,3.70,3.76,3.79,3.90,4.08,4.29,4.87,4.89


****
**Preparing rolling AR forecasts**

In [ ]:
#split the data into different maturities (for AR model)

trainSplit = {}
#maturities_list = []
for col in df.columns:
    #maturities[col] = df[[col]]
    #maturities_list.append(df[[col]])

    curr = df[[col]]
    trainSplit[col] = {"train": curr[curr.index < "2026-04-03"], "test": curr[curr.index >= "2026-04-03"]}

    #removes date as indexed column for training data so statsmodels doesnt complain
    trainSplit[col]["train"] = trainSplit[col]["train"].reset_index(drop=True)

matList = ["0Y1M", "0Y3M", "0Y6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]

testArr = []
for mat in matList:
    testArr.append(trainSplit[mat]["test"][mat].to_numpy())

****
**Rolling AR forecasts**


In [ ]:
# Forecasts are created in the final cell.


19  :  [0.07011398 0.07011398 0.07011398]
15  :  [0.0701355 0.0701355 0.0701355]
23  :  [0.07018227 0.07018227 0.07018227]
18  :  [0.0703236 0.0703236 0.0703236]
5  :  [0.07051766 0.07051766 0.07051766]
20  :  [0.07060877 0.07060877 0.07060877]
4  :  [0.07068152 0.07068152 0.07068152]
16  :  [0.07068375 0.07068375 0.07068375]
7  :  [0.0706899 0.0706899 0.0706899]
17  :  [0.07075218 0.07075218 0.07075218]
24  :  [0.07083554 0.07083554 0.07083554]
6  :  [0.07093285 0.07093285 0.07093285]
3  :  [0.07096654 0.07096654 0.07096654]
21  :  [0.07113084 0.07113084 0.07113084]
22  :  [0.0711311 0.0711311 0.0711311]
25  :  [0.071289 0.071289 0.071289]
1  :  [0.07135059 0.07135059 0.07135059]
14  :  [0.07155681 0.07155681 0.07155681]
8  :  [0.07157028 0.07157028 0.07157028]
9  :  [0.0716225 0.0716225 0.0716225]
26  :  [0.07166562 0.07166562 0.07166562]
27  :  [0.07167985 0.07167985 0.07167985]
2  :  [0.07174356 0.07174356 0.07174356]
13  :  [0.0719776 0.0719776 0.0719776]
10  :  [0.0721799 0.07217

**** 
***Final AR model training and rolling forecasts***

In [ ]:
# create rolling AR(5) forecasts for each 20-day test window
lag = 5
horizon = 20
testDates = trainSplit[matList[0]]["test"].index
windowStarts = testDates[:len(testDates) - horizon + 1]
forecasts = []

for j, startDate in enumerate(windowStarts):
    forecasts.append([])
    trainEnd = df.index < startDate

    for mat in matList:
        train = df.loc[trainEnd, mat].reset_index(drop=True)
        model = AutoReg(
            endog = train,
            lags = lag,
            trend = "c"
        )
        forecasts[j].append(model.fit().forecast(steps=horizon).to_numpy())

# forecasts[j][i][0] is the first horizon for the ith maturity in the jth window

#print(forecasts)


def rmseError(errors):
    return 100* np.sqrt(np.mean(errors**2))

def maeError(errors):
    return np.mean(np.abs(errors)) * 100

# use this for the directional errors
def basisError(errors):
    return 100 * np.mean(errors)

horizons = []
errors = []
rmse = []
for i in range(len(forecasts)):
    horizons.append([])
    errors.append([])
    rmse.append([])

    horizons[i].extend((forecasts[i][0], forecasts[i][4], forecasts[i][19]))
    errors[i].extend((forecasts[i][0] - testArr[i][0], forecasts[i][4] - testArr[i][4], forecasts[i][19] - testArr[i][19]))
    rmse[i].extend((100*np.sqrt(np.mean(errors[i][0]**2)), 100*np.sqrt(np.mean(errors[i][1]**2)), 100*np.sqrt(np.mean(errors[i][2]**2))))

    print("Maturity: ", matList[i], "\nRMSE for 1 Day forecast: ", rmse[i][0], "\nError for 1 Day forecast: ", 100*errors[i][0],
        "\nRMSE for 5 day forecast: ", rmse[i][1], "\nError for 5 day forecast: ", 100*errors[i][1],
        "\nRMSE for 20 day forecast: ", rmse[i][2], "\nError for 20 day forecast: ", 100*errors[i][2],
        "\n\n")    






Maturity:  0Y1M 
RMSE for 1 Day forecast:  0.7431286718488472 
Error for 1 Day forecast:  0.7431286718488472 
RMSE for 5 day forecast:  1.1214294928850954 
Error for 5 day forecast:  1.1214294928850954 
RMSE for 20 day forecast:  3.7150565114568934 
Error for 20 day forecast:  -3.7150565114568934 


Maturity:  0Y3M 
RMSE for 1 Day forecast:  0.028915025097653313 
Error for 1 Day forecast:  0.028915025097653313 
RMSE for 5 day forecast:  1.7479480666144909 
Error for 5 day forecast:  1.7479480666144909 
RMSE for 20 day forecast:  3.8762228732465154 
Error for 20 day forecast:  -3.8762228732465154 


Maturity:  0Y6M 
RMSE for 1 Day forecast:  1.1186367117320017 
Error for 1 Day forecast:  1.1186367117320017 
RMSE for 5 day forecast:  0.007724510200901591 
Error for 5 day forecast:  0.007724510200901591 
RMSE for 20 day forecast:  8.380466802095832 
Error for 20 day forecast:  -8.380466802095832 


Maturity:  1Y 
RMSE for 1 Day forecast:  3.8001896038025063 
Error for 1 Day forecast:  3.8